# Đồ án KHDL cuối kỳ — Chuyên đề 3: Phân tích và gợi ý bất động sản TP.HCM

## Notebook 01: Mô tả bài toán và dữ liệu

- **Học viên**: [Điền tên]
- **Môn**: Lập trình cho Khoa học Dữ liệu
- **Giảng viên hướng dẫn**: [Điền tên giảng viên]
- **Ngày**: 23/07/2026

## 1. Bối cảnh và mục tiêu

Tin đăng bất động sản nhà phố tại TP.HCM thường có:
- **Giá viết không thống nhất** ("3,5 tỷ", "3500 triệu", "Giá thỏa thuận").
- **Diện tích dạng số thập phân với dấu phẩy** ("75,5 m²").
- **Địa danh viết tắt / lẫn lộn** ("Q.1", "Quận 1", "1" — đều là cùng một quận).
- **Nhiều tin đăng trùng** và **ngoại lệ rõ ràng** (giá vài chục triệu, diện tích 7000 m²).

Mục tiêu đồ án:
1. Xây dựng **pipeline chuẩn hóa** giá, diện tích, giá/m², địa danh và thuộc tính mô tả.
2. Phát hiện **tin trùng** và **ngoại lệ**.
3. **Dự đoán** `price_per_m2` từ các đặc trưng (diện tích, phòng ngủ, hướng, quận, tiện ích).
4. **Phân khúc** BĐS bằng K-Means và **gợi ý top 5** theo hồ sơ nhu cầu (ngân sách, số phòng, diện tích, khu vực).

## 2. Sáu câu hỏi nghiên cứu

1. Khu vực nào có giá trên mét vuông (`price_per_m2`) cao nhất?
2. Diện tích và số phòng ảnh hưởng như thế nào đến tổng giá và giá/m²?
3. Hướng nhà, vị trí quận có liên hệ gì với giá không?
4. Tin nào có giá/diện tích bất thường hoặc có khả năng trùng?
5. Mô hình dự đoán giá/m² đạt sai số bao nhiêu (MAE / RMSE / R²)?
6. Top 5 tin nào phù hợp với từng hồ sơ nhu cầu (gia đình trẻ / nhà đầu tư / mua đầu tiên)?

## 3. Hai nguồn dữ liệu

| Nguồn | File | Số dòng | Mô tả |
|---|---|---|---|
| 1 (chính) | `data/raw/real_estate_with_price_per_m2.csv` | 3000 | Tin đăng BĐS nhà phố tại TP.HCM, đã crawl từ nguồn alonhadat-style, có schema 20 cột. |
| 2 (phụ) | `data/raw/neighborhood_amenities.csv` | 100 | Thông tin tiện ích (trường học, bệnh viện, siêu thị, công viên, bus) theo (quận, phường) — tự tổng hợp deterministic. |

## 4. Từ điển dữ liệu (rút gọn)

| Cột | Kiểu | Mô tả |
|---|---|---|
| `listing_id` | int | Mã tin đăng. |
| `title`, `description` | str | Tiêu đề + mô tả. |
| `property_type` | str | Loại hình (toàn bộ là "Nhà" trong dataset này). |
| `province` | str | Tỉnh/TP (toàn bộ là "Hồ Chí Minh"). |
| `district` | str | Quận/huyện (thô — có dạng số "1" và tên "Bình Thạnh"). |
| `ward` | str | Phường/xã (có missing ~6.7%). |
| `street_name`, `project_name` | str | Tên đường, tên dự án (missing ~26.5% / 98.6%). |
| `total_price` | float | Tổng giá (VND). |
| `area_m2` | float | Diện tích (m²). |
| `price_per_m2` | float | Giá/m² tính sẵn (sẽ tính lại để đảm bảo nhất quán). |
| `floor_count`, `frontage_width`, `house_depth`, `road_width` | float | Đặc trưng nhà (nhiều missing). |
| `bedrooms`, `bathrooms` | float | Số phòng. |
| `house_direction` | str | Hướng nhà (8 hướng chính + dạng ghép). |
| `posted_at` | str | Ngày đăng (chuỗi ISO). |

Chi tiết từng cột xem `reports/data_dictionary.md`.

## 5. Dữ liệu mẫu — 5 dòng đầu

In [ ]:
import pandas as pd

raw = pd.read_csv('data/raw/real_estate_with_price_per_m2.csv')
print('Shape:', raw.shape)
raw.head()

## 6. Dữ liệu mẫu — 5 dòng cuối

In [ ]:
raw.tail()

## 7. Schema và dtype

In [ ]:
raw.info()

## 8. Tỉ lệ missing theo cột

In [ ]:
(raw.isna().mean() * 100).round(2).sort_values(ascending=False)

## 9. Kết luận

Dữ liệu đủ lớn (3000 dòng) và đa dạng (22 quận/huyện, 144 phường/xã, nhiều mức giá).
Các cột có missing > 80% (`house_depth`, `project_name`, `road_width`) sẽ **không** dùng làm feature cho mô hình, chỉ giữ để tham khảo khi cần.

Tiếp theo: mở `02_collection_and_cleaning.ipynb` để xem pipeline làm sạch chi tiết.